In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import requests
from langchain_core.messages import HumanMessage,ToolMessage,AIMessage,SystemMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

c:\Users\Keshab\anaconda3\envs\Learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id = "mistralai/Mistral-7B-Instruct-v0.2",
    task = "text-generation",
    # provider="hf-inference" 

)
llm = ChatHuggingFace(llm=llm)

## Tool create

In [3]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str) ->float:
    """This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url=f'https://v6.exchangerate-api.com/v6/f5ae4b2463745e6681fc56ef/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()
@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """given a currency conversion rate this function calculates the target currency value from a given currency value"""
    return base_currency_value*conversion_rate



In [4]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772496002,
 'time_last_update_utc': 'Tue, 03 Mar 2026 00:00:02 +0000',
 'time_next_update_unix': 1772582402,
 'time_next_update_utc': 'Wed, 04 Mar 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 91.6326}

In [5]:
convert.invoke({'base_currency_value':10,'conversion_rate':91.6326})

916.326

### tool binding

In [20]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [21]:
message=[HumanMessage('what is the conversion factor between USD and INR ,and based on that can you convert 10 USD into INR')]

In [22]:
message

[HumanMessage(content='what is the conversion factor between USD and INR ,and based on that can you convert 10 USD into INR', additional_kwargs={}, response_metadata={})]

In [23]:
ai_message = llm_with_tools.invoke(message)

In [24]:
message.append(ai_message)

In [15]:
print(ai_message)

content=" As of February 13, 2023, the conversion rate between 1 US Dollar (USD) and 1 Indian Rupee (INR) is approximately 75.25 INR. Therefore, to convert 10 USD to INR, you would multiply 10 by 75.25:\n\n10 USD * 75.25 INR/USD = 752.5 INR \n\nSo, 10 USD is approximately equal to 752.5 INR. However, keep in mind that exchange rates can fluctuate daily, so it's always a good idea to check the latest conversion rate before making a transaction." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 33, 'total_tokens': 188}, 'model_name': 'mistralai/Mistral-7B-Instruct-v0.2', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cb2f6-5704-76d1-bad4-8a7649f51798-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 33, 'output_tokens': 155, 'total_tokens': 188}


In [25]:
ai_message.tool_calls

[]

In [26]:
import json
for tool_call in ai_message.tool_calls:
    #execute the 1st tool and get the value of conversion rate 
    if tool_call['name']=='get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        #fetch this conversion rate 
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to message list
        message.append(tool_message1)
    #execute teh 2nd tool using the conversion rate from toll 1
    if tool_call['name']=='convert':
        #fetch the current arg 
        tool_call['args']['convesion_rate']==conversion_rate
        tool_message2=convert.invoke(tool_call)
        message.append(tool_message2)

In [27]:
message

[HumanMessage(content='what is the conversion factor between USD and INR ,and based on that can you convert 10 USD into INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content=" As of February 1, 2023, the conversion rate between 1 US Dollar (USD) and 1 Indian Rupee (INR) is approximately 75.25 INR (Indian Rupees). Therefore, to convert 10 US Dollars to Indian Rupees, you can multiply the number of US Dollars by the conversion factor:\n\n10 USD * 75.25 INR/USD = 752.5 INR\n\nSo, 10 US Dollars is equal to approximately 752.5 Indian Rupees. Please note that exchange rates fluctuate daily, so it's essential to check the current rate before making a conversion.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 33, 'total_tokens': 195}, 'model_name': 'mistralai/Mistral-7B-Instruct-v0.2', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cb303-e063-77a2-9423-0c623c132084-0', tool_calls=[], in

In [ ]:
llm_with_tools.invoke(message).content